# MiniBioDesignBench Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wenxy59/program26summer/blob/main/notebooks/MiniBioDesignBench_Colab.ipynb)

This notebook lets students run the MiniBioDesignBench demo in Google Colab without setting up a local machine or logging into a remote server.

**How to use**

1. Click **Runtime -> Run all**.
2. If Colab asks for permission to run cells, approve it.
3. The demo should finish in under one minute on the default CPU runtime.

The project does **not** require GPU, CUDA, conda, OpenAI keys, or server access.

## 0. What is Colab?

Google Colab is a browser-based Jupyter notebook environment. Each notebook runs inside a temporary Linux machine managed by Google.

Useful mental model:

- A **cell** is a small block of Markdown or Python/shell code.
- `!command` runs a Linux shell command, such as `!ls` or `!python3 run_demo.py`.
- The runtime is temporary. If it disconnects, rerun the setup cells.
- Save important edits by downloading files, saving a copy to Drive, or committing to GitHub.

## 1. Check the Colab runtime

This project only needs CPU. GPU is optional and unnecessary for the deterministic demo.

In [ ]:
import os
import platform
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Current directory:", os.getcwd())

In [ ]:
# Optional: check whether Colab attached a GPU.
# It is fine if this prints "No GPU found".
!nvidia-smi || echo "No GPU found; CPU runtime is enough for this demo."

## 2. Clone the project from GitHub

Colab starts from a clean temporary machine, so we clone the repository every time.

In [ ]:
REPO_URL = "https://github.com/wenxy59/program26summer.git"
PROJECT_DIR = "/content/program26summer"

!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!find . -maxdepth 2 -type f | sort

## 3. Run the required smoke test

The gold submissions are known-valid answers. If this cell passes, the dataset and grader are working.

In [ ]:
!python3 run_demo.py --mode grade-gold

Expected final line:

```text
Gold submissions: 8/8 pass
```

## 4. Run all deterministic demos

This compares direct, tool-using, and repair baselines, then prints Pareto fronts for candidate-selection tasks.

In [ ]:
!python3 run_demo.py --mode all

## 5. Inspect the dataset

Each task is stored as JSON. Students should read the task prompt, hard constraints, soft objectives, and the gold answer.

In [ ]:
import json
from pathlib import Path

tasks = json.loads(Path("data/tasks.json").read_text())
gold = json.loads(Path("data/gold_submissions.json").read_text())

print("Number of tasks:", len(tasks))
for i, task in enumerate(tasks, 1):
    objectives = [obj.get("name", obj.get("objective_id", "objective")) for obj in task.get("soft_objectives", [])]
    print(f"{i:02d}. {task['task_id']} | level={task.get('level')} | soft_objectives={objectives}")

In [ ]:
# Pick one task and inspect it in detail.
TASK_INDEX = 0
task = tasks[TASK_INDEX]
task_id = task["task_id"]

print("TASK")
print(json.dumps(task, indent=2))
print("\nGOLD SUBMISSION")
print(json.dumps(gold[task_id], indent=2))

## 6. Student exercise: failure analysis

Try changing `TASK_INDEX` above and answer:

1. What are the hard constraints?
2. What are the soft objectives?
3. What would a naive LLM likely get wrong?
4. How could a repair agent use grader feedback to improve?

## 7. Optional: edit tasks in Colab

Colab's file browser is on the left sidebar. Open `data/tasks.json` or `data/gold_submissions.json`, edit the file, then rerun the smoke test.

Because the runtime is temporary, download your edited files before closing Colab.

In [ ]:
# Optional helper: zip the current working copy for download.
# Run this after making edits in Colab.
from google.colab import files

!zip -r minibio_colab_work.zip README.md run_demo.py data src requirements.txt LICENSE
files.download("minibio_colab_work.zip")